# Protocols in Python

This module covers Python's protocols: the conventions by which objects plug into language constructs such as `for`, `with`, `len()`, `in`, `+`, attribute access, and similar. Readers are assumed to be comfortable with classes and dunders (see `classes.md`); the focus here is on which protocols matter in practice, how to implement each one correctly, and when reaching for a protocol is the right design move rather than an over-engineered one.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. What a protocol is
2. The iteration protocol
3. Iterables vs. iterators
4. Generators as iterators
5. The container protocol
6. Subscription
7. The context manager protocol
8. The callable protocol
9. String conversion and formatting
10. Truthiness
11. Equality and hashing
12. Ordering
13. Numeric operations
14. Attribute access as a protocol
15. Descriptors
16. Asynchronous counterparts
17. `collections.abc` and protocol mixins
18. `typing.Protocol` and structural typing
19. Real-world design principles
20. Common mistakes

---

## 1. What a protocol is

A protocol in Python is an informal interface defined by which special methods (dunders) an object implements. The interpreter drives most language constructs through these methods: `for x in obj` looks up `obj.__iter__`, `len(obj)` looks up `obj.__len__`, `with obj as x` looks up `obj.__enter__` and `obj.__exit__`, `obj()` looks up `obj.__call__`. Any object that defines the right methods participates in the construct, regardless of declared type or class hierarchy.

This is the runtime expression of the duck typing introduced in `classes.md` Topic 16. Languages such as Java express the same idea through interfaces and `implements` clauses; Python places the contract in the methods themselves. There is no `implements Iterable` keyword; defining `__iter__` is the entire commitment.

Two consequences flow from this design. First, almost every operator and built-in function in the language delegates to a method that can be defined on a class. Second, an object can satisfy several protocols at once with no declaration: a single class can be iterable, sized, hashable, callable, and a context manager simultaneously, simply by implementing the relevant methods.

In [ ]:
class TransactionLog:
    def __init__(self) -> None:
        self.entries: list[tuple[str, float]] = []

    def __len__(self) -> int:
        return len(self.entries)

    def __iter__(self):
        return iter(self.entries)

    def __contains__(self, kind: str) -> bool:
        return any(entry[0] == kind for entry in self.entries)


log = TransactionLog()
log.entries.extend([("deposit", 100.0), ("withdraw", 30.0)])

len(log)              # 2 — uses __len__
list(log)             # [('deposit', 100.0), ('withdraw', 30.0)] — uses __iter__
"deposit" in log      # True — uses __contains__

`TransactionLog` declares no base class. It earns participation in `len`, iteration, and `in` purely by defining the matching methods. The rest of this module walks through the protocols that come up most often in real Python code.

## 2. The iteration protocol

`for` loops, comprehensions, `*`-unpacking in calls, generator expressions, and the `iter()` and `next()` built-ins all rest on the same two-method protocol. An iterator is any object that defines `__next__` (returning the next value or raising `StopIteration`) and `__iter__` (returning the iterator itself). An iterable is any object whose `__iter__` returns an iterator.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[tuple[str, float]] = []

    def __iter__(self):
        return iter(self.transactions)


account = Account("Anna Smith", 500.0)
account.transactions.extend([("deposit", 100.0), ("withdraw", 30.0)])

for kind, amount in account:
    print(kind, amount)
# deposit 100.0
# withdraw 30.0

list(account)                      # [('deposit', 100.0), ('withdraw', 30.0)]
first = next(iter(account))        # ('deposit', 100.0)
deposits, withdrawals = (         # unpacking via iteration
    [t for t in account if t[0] == "deposit"],
    [t for t in account if t[0] == "withdraw"],
)

`for` does not call `__next__` directly. It first calls `iter(account)`, which delegates to `account.__iter__()`, then repeatedly calls `__next__` on the returned iterator, stopping when `StopIteration` is raised. The `StopIteration` is caught by the loop machinery and never surfaces to the caller.

This indirection is what allows the same object to be iterated repeatedly. Each `for` over `account` constructs a fresh iterator over `self.transactions`. If `__iter__` returned `self` and consumed state, a second loop would silently see nothing.

## 3. Iterables vs. iterators

The two roles look similar but should not be conflated. An iterable is a source: asking it for an iterator must produce a new one each time. An iterator is single-use: it tracks position and is exhausted after the last value.

In [ ]:
numbers = [1, 2, 3]            # iterable
it = iter(numbers)             # iterator (single use)

list(it)                       # [1, 2, 3]
list(it)                       # [] — already exhausted

list(numbers)                  # [1, 2, 3]
list(numbers)                  # [1, 2, 3] — fresh iterator each time

Built-in collections (`list`, `tuple`, `set`, `dict`) are iterables; they can be looped over many times. `iter()`, `enumerate()`, `zip()`, `map()`, `filter()`, generator expressions, and file objects return iterators; they can be looped over once.

A class that implements iteration directly should usually be an iterable, not an iterator. The standard pattern is to keep state outside the iteration object:

In [ ]:
class TransactionLog:
    def __init__(self) -> None:
        self.entries: list[tuple[str, float]] = []

    def __iter__(self):
        return iter(self.entries)         # fresh iterator each call


log = TransactionLog()
log.entries.extend([("deposit", 100.0), ("withdraw", 30.0)])

for entry in log: ...                     # works
for entry in log: ...                     # works again

Writing `__iter__` to `return self` and adding `__next__` produces an iterator. That is the right design for objects whose purpose is to advance through a sequence (a database cursor, a paginated API reader) but the wrong design for collections.

## 4. Generators as iterators

A function containing `yield` is a generator function. Calling it returns a generator object, which is an iterator: it implements `__iter__` (returning itself) and `__next__` (resuming the function until the next `yield` or until it returns, which raises `StopIteration`). This is by far the simplest way to produce an iterator without writing a class.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[tuple[str, float]] = []

    def deposits(self):
        for kind, amount in self.transactions:
            if kind == "deposit":
                yield amount

    def __iter__(self):
        return iter(self.transactions)


account = Account("Anna Smith", 500.0)
account.transactions.extend(
    [("deposit", 100.0), ("withdraw", 30.0), ("deposit", 50.0)]
)

list(account.deposits())          # [100.0, 50.0]
sum(account.deposits())           # 150.0

`account.deposits()` returns a fresh generator each time it is called, so iteration can be repeated without surprises. Generators are lazy: values are produced one at a time on demand, which matters when filtering or transforming long sequences where a materialized list would waste memory.

Generator expressions (`(amount for kind, amount in account.transactions if kind == "deposit")`) are the same machinery in expression form. They are preferred over list comprehensions when the result is consumed once and never indexed.

## 5. The container protocol

The container protocol covers `len(obj)`, `x in obj`, and the index-form access discussed in the next topic. The relevant methods are `__len__` and `__contains__`.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[tuple[str, float]] = []

    def __len__(self) -> int:
        return len(self.transactions)

    def __contains__(self, kind: str) -> bool:
        return any(entry[0] == kind for entry in self.transactions)


account = Account("Anna Smith", 500.0)
account.transactions.extend([("deposit", 100.0), ("withdraw", 30.0)])

len(account)              # 2
"deposit" in account      # True
"transfer" in account     # False

If `__contains__` is not defined, `in` falls back to iteration: it calls `__iter__` and compares each element with `==`. This is why `5 in [1, 2, 3, 4, 5]` works on a list even though list does not register a special membership method for every integer. Defining `__contains__` explicitly is worthwhile when membership can be answered faster than O(n) (a set's `__contains__` is O(1)) or when the question being asked is not "is this exact value in the iteration."

`__len__` has a second role: it is consulted by `bool()` when `__bool__` is not defined (Topic 10). An empty container is therefore falsy without any extra work.

## 6. Subscription

Index and key access — `obj[key]`, `obj[key] = value`, `del obj[key]` — go through `__getitem__`, `__setitem__`, and `__delitem__`. The key can be anything: an integer, a string, a tuple, a slice. The object decides what is meaningful.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[tuple[str, float]] = []

    def __getitem__(self, index):
        return self.transactions[index]      # supports both int and slice

    def __setitem__(self, index: int, value: tuple[str, float]) -> None:
        self.transactions[index] = value

    def __delitem__(self, index: int) -> None:
        del self.transactions[index]


account = Account("Anna Smith", 500.0)
account.transactions.extend(
    [("deposit", 100.0), ("withdraw", 30.0), ("deposit", 50.0)]
)

account[0]                # ('deposit', 100.0)
account[-1]               # ('deposit', 50.0)
account[1:]               # [('withdraw', 30.0), ('deposit', 50.0)]
account[1] = ("fee", 5.0)
del account[0]

For mapping-style access, the same methods take a string or other hashable key. A `Bank` keyed by account_id would define `__getitem__` to look up by ID, and `__setitem__` to register a new account.

A historical quirk: if a class defines `__getitem__` but not `__iter__`, Python falls back to a legacy iteration protocol that calls `obj[0]`, `obj[1]`, `obj[2]`, … until `IndexError` is raised. This is why old-style sequence classes can be looped over without `__iter__`. New code should define `__iter__` explicitly when iteration is intended.

## 7. The context manager protocol

A context manager is any object implementing `__enter__(self)` and `__exit__(self, exc_type, exc_value, traceback)`. The `with` statement enters the manager, binds the result of `__enter__` to the `as` variable, runs the body, then calls `__exit__` regardless of how the body exited.

In [ ]:
import time


class TransactionBatch:
    def __init__(self, account: "Account") -> None:
        self.account = account
        self.staged: list[tuple[str, float]] = []
        self.start: float = 0.0

    def __enter__(self) -> "TransactionBatch":
        self.start = time.perf_counter()
        return self

    def add(self, kind: str, amount: float) -> None:
        self.staged.append((kind, amount))

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            self.account.transactions.extend(self.staged)
        elapsed = time.perf_counter() - self.start
        print(f"batch took {elapsed:.3f}s, committed={exc_type is None}")
        return False         # do not suppress exceptions


with TransactionBatch(account) as batch:
    batch.add("deposit", 200.0)
    batch.add("withdraw", 50.0)
# committed only if the block raised no exception

`__exit__` returning a truthy value suppresses the exception that propagated out of the block; returning a falsy value (including `None`) lets it continue. The context manager protocol is one of the most commonly implemented protocols in tm1py work and is covered in depth in `context_managers.md`, including `contextlib.contextmanager`, `ExitStack`, and the asynchronous variants.

## 8. The callable protocol

Anything that can be invoked with `obj(...)` is callable. Functions, methods, classes, lambdas, and any instance whose class defines `__call__` qualify. Defining `__call__` lets an object hold state across invocations while still being usable wherever a function is expected.

In [ ]:
class WithdrawalLimit:
    def __init__(self, daily_max: float) -> None:
        self.daily_max = daily_max
        self.withdrawn_today: float = 0.0

    def __call__(self, amount: float) -> bool:
        if self.withdrawn_today + amount > self.daily_max:
            return False
        self.withdrawn_today += amount
        return True


limit = WithdrawalLimit(daily_max=500.0)

limit(200.0)              # True
limit(250.0)              # True
limit(100.0)              # False — would exceed 500
callable(limit)           # True

`callable(obj)` checks for `__call__`. Because a class's `__init__` runs as part of calling the class, `callable(SomeClass)` is `True`; the result of calling it is an instance.

Callable objects are the natural choice when a function would need to carry significant configuration: validators, dispatchers, retry policies, fitted machine-learning models. A bare function with closures can do the same job, but a class is easier to inspect, pickle, and extend through subclassing.

## 9. String conversion and formatting

Three protocols govern how an object becomes a string. `__repr__` is the developer-facing form, ideally an unambiguous reconstruction string. `__str__` is the user-facing form, allowed to be friendlier and lossier. `__format__` controls behavior under `format(obj, spec)` and inside f-strings (`f"{obj:spec}"`).

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def __repr__(self) -> str:
        return f"Account(owner={self.owner!r}, balance={self.balance!r})"

    def __str__(self) -> str:
        return f"{self.owner}: {self.balance:.2f} EUR"

    def __format__(self, spec: str) -> str:
        if spec == "short":
            return f"{self.owner.split()[0]}/{self.balance:.0f}"
        if spec == "":
            return str(self)
        raise ValueError(f"unknown format spec for Account: {spec!r}")


account = Account("Anna Smith", 500.0)

repr(account)             # "Account(owner='Anna Smith', balance=500.0)"
str(account)              # 'Anna Smith: 500.00 EUR'
f"{account}"              # 'Anna Smith: 500.00 EUR' — uses __str__
f"{account:short}"        # 'Anna/500' — uses __format__ with spec='short'

If `__str__` is not defined, `str()` falls back to `__repr__`. If `__format__` is not defined, the default behavior on a non-empty spec raises `TypeError`, and on an empty spec it calls `__str__`. Implementing `__format__` is uncommon outside of types meant for templating (numbers, dates, currencies), but it is the right hook when an f-string would otherwise need a dedicated method call.

## 10. Truthiness

Every Python object can be tested as a boolean. The protocol is `__bool__`, which must return a `bool`. If `__bool__` is not defined, Python falls back to `__len__`: an object with length 0 is falsy, anything else is truthy. If neither is defined, the object is unconditionally truthy.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[tuple[str, float]] = []

    def __bool__(self) -> bool:
        return self.balance > 0


a = Account("Anna Smith", 500.0)
b = Account("Ben Jones", 0.0)

bool(a)                   # True
bool(b)                   # False
if a:
    print("has funds")    # printed
if not b:
    print("empty")        # printed

The fallback to `__len__` is the reason empty lists, dicts, and strings are falsy without anyone writing `__bool__` for them. It is also a small trap: a custom collection class that defines `__len__` becomes implicitly falsy when empty, which is usually desired but worth knowing.

Defining `__bool__` is appropriate when "is this object meaningful?" has a domain answer that is not equivalent to "does it have anything in it?". A `Match` object from `re.match`, for example, is truthy when a match was found and falsy otherwise, regardless of any contained content.

## 11. Equality and hashing

Equality is `__eq__`, hashing is `__hash__`. These two are bound together by a contract: objects that compare equal must hash equal. The reverse is not required (hash collisions are allowed), but violating the forward direction breaks `dict` and `set`.

In [ ]:
class Account:
    def __init__(self, account_id: str, owner: str, balance: float = 0.0) -> None:
        self.account_id = account_id
        self.owner = owner
        self.balance = balance

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return self.account_id == other.account_id

    def __hash__(self) -> int:
        return hash(self.account_id)


a = Account("AC-001", "Anna Smith", 500.0)
b = Account("AC-001", "Anna Smith", 999.0)
c = Account("AC-002", "Ben Jones", 100.0)

a == b                    # True — same account_id
a == c                    # False
{a, b, c}                 # {<AC-001>, <AC-002>} — a and b collapse

Returning `NotImplemented` (not `False`) when the other object is of an unrelated type is what allows Python to try the reflected operation. If `Account.__eq__` is asked to compare against a `str` and returns `NotImplemented`, Python will then ask the `str` whether it knows how to compare with an `Account`. Returning `False` short-circuits this and produces wrong answers for symmetric comparisons.

Defining `__eq__` without `__hash__` makes the class unhashable: Python silently sets `__hash__` to `None`. This is intentional, because mutable objects whose hashable fields can change would corrupt any container holding them. If hashing is wanted, define `__hash__` explicitly and base it on attributes that do not change.

## 12. Ordering

The comparison protocol is `__lt__`, `__le__`, `__gt__`, `__ge__`, in addition to `__eq__` and `__ne__`. Python uses these for `<`, `<=`, `>`, `>=`, and indirectly for `sorted()`, `min()`, `max()`, and `heapq`.

In [ ]:
from functools import total_ordering


@total_ordering
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return self.balance == other.balance

    def __lt__(self, other: "Account") -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return self.balance < other.balance


accounts = [
    Account("Anna Smith", 500.0),
    Account("Ben Jones", 1000.0),
    Account("Cara Lopez", 250.0),
]

sorted(accounts)[0].owner       # 'Cara Lopez'
max(accounts).owner             # 'Ben Jones'

`@total_ordering` fills in the remaining three comparison methods (`__le__`, `__gt__`, `__ge__`) given `__eq__` and one of `__lt__`/`__le__`/`__gt__`/`__ge__`. Without it, only the operators that have been explicitly defined will work, and the rest will raise `TypeError`.

When sorting by something other than the natural order, `sorted(items, key=...)` is preferred over implementing `__lt__` to switch behavior. The protocol is for the canonical order of the type; ad hoc orders belong in the `key` argument.

## 13. Numeric operations

Arithmetic operators bind to a family of dunders: `__add__` for `+`, `__sub__` for `-`, `__mul__` for `*`, `__truediv__` for `/`, and so on through `__floordiv__`, `__mod__`, `__pow__`, `__neg__`, `__abs__`, `__round__`. Each binary operator has a reflected variant (`__radd__`, `__rsub__`, …) used when the left operand does not handle the operation, and an in-place variant (`__iadd__`, `__isub__`, …) used by `+=` and `-=`.

In [ ]:
class Money:
    def __init__(self, amount: float, currency: str = "EUR") -> None:
        self.amount = amount
        self.currency = currency

    def __repr__(self) -> str:
        return f"Money({self.amount!r}, {self.currency!r})"

    def __add__(self, other: object) -> "Money":
        if isinstance(other, Money) and other.currency == self.currency:
            return Money(self.amount + other.amount, self.currency)
        if isinstance(other, (int, float)):
            return Money(self.amount + other, self.currency)
        return NotImplemented

    def __radd__(self, other: object) -> "Money":
        return self.__add__(other)        # 100 + Money(...) routes here

    def __mul__(self, factor: object) -> "Money":
        if isinstance(factor, (int, float)):
            return Money(self.amount * factor, self.currency)
        return NotImplemented

    def __neg__(self) -> "Money":
        return Money(-self.amount, self.currency)


m = Money(100.0)
m + Money(50.0)           # Money(150.0, 'EUR')
m + 25                    # Money(125.0, 'EUR')
25 + m                    # Money(125.0, 'EUR') — uses __radd__
m * 3                     # Money(300.0, 'EUR')
-m                        # Money(-100.0, 'EUR')
sum([Money(10.0), Money(20.0)])   # Money(30.0, 'EUR') — sum starts from 0

`sum()` starts from `0` and repeatedly applies `+`, which is why `__radd__` (handling `0 + Money(...)`) is needed for `sum` to work on a list of `Money`. Returning `NotImplemented` for unsupported operand types is the convention; raising `TypeError` directly defeats the reflected-operator machinery.

In-place variants are an optimization for mutable types: `list.__iadd__` extends the list in place, while a fresh `+` would allocate a new list. For immutable types like `Money`, omitting `__iadd__` is fine; Python falls back to `__add__` followed by rebinding.

## 14. Attribute access as a protocol

Reading, writing, and deleting attributes go through dunders that can be intercepted: `__getattribute__` is called for every attribute access; `__getattr__` is called only when normal lookup fails; `__setattr__` and `__delattr__` are called for every assignment and deletion. `__dir__` controls what `dir(obj)` reports.

`__getattr__` is the most commonly useful of these because it is non-invasive: it only runs when an attribute is missing, so it cannot accidentally interfere with normal access.

In [ ]:
class LazyAccount:
    def __init__(self, account_id: str, loader) -> None:
        self.account_id = account_id
        self._loader = loader
        self._loaded: dict | None = None

    def __getattr__(self, name: str):
        if self._loaded is None:
            self._loaded = self._loader(self.account_id)
        if name in self._loaded:
            return self._loaded[name]
        raise AttributeError(name)


def fetch(account_id: str) -> dict:
    return {"owner": "Anna Smith", "balance": 500.0}


lazy = LazyAccount("AC-001", fetch)
lazy.account_id           # 'AC-001' — present, __getattr__ not consulted
lazy.owner                # 'Anna Smith' — triggers fetch, served from _loaded
lazy.balance              # 500.0 — _loaded is reused

`__getattribute__` is more aggressive: it runs for _every_ access, including `self._loaded`, so any implementation must call `super().__getattribute__(name)` for the attributes it does not want to override. Mistakes here cause infinite recursion. Reach for `__getattr__` first; only escalate to `__getattribute__` when interception of even existing attributes is unavoidable.

`__setattr__` is occasionally useful for write-time validation or for read-only objects:

In [ ]:
class FrozenAccount:
    def __init__(self, owner: str, balance: float) -> None:
        object.__setattr__(self, "owner", owner)
        object.__setattr__(self, "balance", balance)

    def __setattr__(self, name: str, value) -> None:
        raise AttributeError(f"FrozenAccount is immutable")


a = FrozenAccount("Anna Smith", 500.0)
a.balance = 0.0           # AttributeError

`object.__setattr__(self, ...)` bypasses the override, which is the standard escape hatch when the constructor itself needs to set fields.

## 15. Descriptors

A descriptor is an object that defines any of `__get__`, `__set__`, or `__delete__` and is stored as a class attribute (not an instance attribute). The descriptor protocol is what makes `@property`, `@classmethod`, `@staticmethod`, and bound methods work; understanding it explains a great deal of how Python classes behave.

In [ ]:
class PositiveNumber:
    def __set_name__(self, owner, name: str) -> None:
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self                    # access on the class
        return getattr(instance, self.private_name)

    def __set__(self, instance, value: float) -> None:
        if value < 0:
            raise ValueError("must be non-negative")
        setattr(instance, self.private_name, value)


class Account:
    balance = PositiveNumber()

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance             # routes through PositiveNumber.__set__


account = Account("Anna Smith", 500.0)
account.balance           # 500.0 — routes through PositiveNumber.__get__
account.balance = 200.0   # validation runs
account.balance = -50.0   # ValueError

`__set_name__` (Python 3.6+) is called when the descriptor is bound to a class attribute; it receives the owning class and the attribute name, which is how the descriptor learns where to stash its private value.

A descriptor that defines `__set__` (or `__delete__`) is a _data descriptor_; one that only defines `__get__` is a _non-data descriptor_. Data descriptors take precedence over instance `__dict__` entries; non-data descriptors are shadowed by instance entries. This is why `@property` (a data descriptor) cannot be overridden per-instance, while plain methods (non-data descriptors) can be replaced by assigning to `instance.method_name`.

For most application code, properties are sufficient. Custom descriptors are reached for when the same validation or computed-attribute pattern needs to be reused across many fields or many classes — a typical example is an ORM where every model field is a descriptor.

## 16. Asynchronous counterparts

Each protocol that involves blocking or yielding control has an `async` counterpart. Iteration becomes `__aiter__` and `__anext__`, used by `async for`. Context management becomes `__aenter__` and `__aexit__`, used by `async with`. A coroutine function (`async def`) is the asynchronous analogue of a generator function.

In [ ]:
import asyncio


class AsyncTransactions:
    def __init__(self, account_ids: list[str]) -> None:
        self.account_ids = account_ids
        self.index = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.index >= len(self.account_ids):
            raise StopAsyncIteration
        await asyncio.sleep(0.01)         # simulated network call
        result = (self.account_ids[self.index], 500.0)
        self.index += 1
        return result


async def main():
    async for account_id, balance in AsyncTransactions(["AC-001", "AC-002"]):
        print(account_id, balance)


asyncio.run(main())

The shape mirrors the synchronous protocol: `StopAsyncIteration` ends the loop, `__aiter__` returns the async iterator. Synchronous iterables and asynchronous iterables are not interchangeable: `for` will not accept an `__aiter__`-only object, and `async for` will not accept an `__iter__`-only one.

Most application code does not need to implement the async protocols directly; libraries such as `aiohttp`, `asyncpg`, and modern HTTP clients return objects that already do. Implementing them by hand is appropriate when wrapping an inherently asynchronous source (a websocket stream, a paginated async API).

## 17. `collections.abc` and protocol mixins

The `collections.abc` module provides abstract base classes that formalize Python's structural protocols: `Iterable`, `Iterator`, `Container`, `Sized`, `Collection`, `Sequence`, `MutableSequence`, `Mapping`, `MutableMapping`, `Set`, `Hashable`, `Callable`, plus their async cousins. Two benefits follow from inheriting from one:

First, `isinstance(obj, Iterable)` works structurally — any object with `__iter__` is recognized as an `Iterable` even without explicit inheritance, because the ABCs use `__subclasshook__` to check methods rather than requiring registration.

In [ ]:
from collections.abc import Iterable, Sized

isinstance([1, 2, 3], Iterable)        # True
isinstance([1, 2, 3], Sized)           # True

Second, inheriting from a richer ABC like `Sequence` or `MutableMapping` provides _mixin methods_ for free. `Sequence` requires only `__getitem__` and `__len__`; the ABC supplies `__iter__`, `__contains__`, `__reversed__`, `index`, and `count` based on those two.

In [ ]:
from collections.abc import Sequence


class TransactionLog(Sequence):
    def __init__(self) -> None:
        self.entries: list[tuple[str, float]] = []

    def __getitem__(self, index):
        return self.entries[index]

    def __len__(self) -> int:
        return len(self.entries)


log = TransactionLog()
log.entries.extend([("deposit", 100.0), ("withdraw", 30.0), ("deposit", 50.0)])

list(reversed(log))                    # mixin-supplied
log.count(("deposit", 100.0))          # 1 — mixin-supplied
log.index(("withdraw", 30.0))          # 1 — mixin-supplied
"deposit" in [k for k, _ in log]

For a custom collection class with non-trivial behavior, subclassing the matching ABC is the lowest-effort way to get correct, well-tested implementations of the surrounding methods. The trade-off is a slightly heavier MRO and the assumption that `__getitem__`/`__len__` are reasonably efficient, since the mixins call them.

## 18. `typing.Protocol` and structural typing

`typing.Protocol` (PEP 544) brings static structural typing to Python's type checkers. A `Protocol` subclass is a type definition that says "anything with this shape," and a checker like `mypy` or `pyright` will accept any class whose methods and attributes match, with no inheritance required.

In [ ]:
from typing import Protocol


class HasBalance(Protocol):
    balance: float

    def deposit(self, amount: float) -> None: ...


def total(accounts: list[HasBalance]) -> float:
    return sum(account.balance for account in accounts)


class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        self.balance += amount


total([Account("Anna", 500.0), Account("Ben", 300.0)])    # 800.0 — type-checks

`Account` does not inherit from `HasBalance`, but it satisfies the protocol structurally: it has a `balance` attribute and a `deposit` method with the right signature. The type checker will accept it; at runtime the call works because Python does not enforce types.

For runtime-checkable protocols, decorate with `@runtime_checkable` and use `isinstance`. Runtime checks only verify method names, not signatures or attribute types, so they are weaker than the static check.

In [ ]:
from typing import runtime_checkable


@runtime_checkable
class HasBalance(Protocol):
    balance: float


isinstance(Account("Anna", 500.0), HasBalance)            # True

`Protocol` is the recommended way to write structural type hints for new code. The `collections.abc` ABCs (Topic 17) and `Protocol` overlap in purpose; `Protocol` is more flexible but exists only for typing, while ABCs exist at runtime and provide mixin methods.

## 19. Real-world design principles

Once the mechanics are clear, the practical question is _when_ to implement a protocol and _which one_. A handful of guidelines apply broadly.

**Implement a protocol when the language construct expresses what the object actually means.** A `TransactionLog` that holds a sequence of entries _is_ a sequence; defining `__iter__` and `__len__` lets the rest of the codebase use `for`, `len()`, and `in` instead of `log.entries.foo()` everywhere. Conversely, a `BankReportGenerator` is not a sequence; making it iterable to expose intermediate rows would be misleading. The test is whether a reader, on seeing `for x in obj`, would correctly predict what `x` is.

**Prefer the smallest sufficient protocol.** If callers only need to iterate, define `__iter__` and stop. Adding `__len__`, `__contains__`, `__getitem__`, `__setitem__`, and so on without a clear use case turns a purpose-built class into a half-formed `list` clone, with all the obligations and none of the testing. Real `list` is a better `list` than any custom one will be.

**Match the convention exactly.** Returning `NotImplemented` (not `False`) from `__eq__`, raising `StopIteration` (not `IndexError`) from `__next__`, returning a truthy value from `__exit__` to suppress an exception — these contracts are not negotiable. The interpreter and built-in functions encode them; deviating breaks integrations in non-obvious ways.

**Use `Protocol` to declare intent in type hints; use `collections.abc` to inherit behavior.** A function that takes "anything iterable" should annotate `Iterable[T]`, which works whether the caller passes a list, a generator, or a custom class. A class that wants the mixin methods of `Sequence` should inherit from `collections.abc.Sequence`. The two serve different needs and the choice is rarely ambiguous in practice.

**Reach for descriptors only when properties stop scaling.** A single validated attribute is a property; ten validated attributes across five classes is a descriptor. The line is fuzzy but the direction is clear: properties first, descriptors when the duplication becomes obvious.

**Do not implement protocols for show.** A class that defines `__call__`, `__iter__`, `__len__`, `__contains__`, `__getitem__`, `__enter__`, `__exit__`, `__add__`, and `__hash__` because each one might be useful is harder to read and harder to change than one that defines exactly the protocols its callers exercise. Each implemented protocol is a public commitment.

## 20. Common mistakes

A handful of errors are easy to make and worth recognizing early.

**Returning `False` instead of `NotImplemented` from `__eq__` or arithmetic dunders.** Returning `NotImplemented` lets Python try the operation on the other operand; returning `False` short-circuits this and produces wrong answers for symmetric comparisons.

In [ ]:
# Wrong
class Money:
    def __eq__(self, other):
        return isinstance(other, Money) and self.amount == other.amount and \
               self.currency == other.currency

# Correct
class Money:
    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

The wrong version returns `False` for `Money(100.0) == "anything"`, which suppresses any chance for the right-hand side to participate. The correct version returns `NotImplemented`, leaving the comparison machinery free to ask the other operand.

**Defining `__eq__` and forgetting `__hash__`.** Once `__eq__` is overridden, Python sets `__hash__` to `None`, making instances unusable as dict keys or set members. This is intentional but easy to miss when the class was previously hashable.

In [ ]:
# Wrong: silently unhashable
class Account:
    def __init__(self, account_id, balance):
        self.account_id = account_id
        self.balance = balance

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented
        return self.account_id == other.account_id


{Account("AC-001", 500.0)}      # TypeError: unhashable type

# Correct: define __hash__ on the same fields used in __eq__
class Account:
    def __init__(self, account_id, balance):
        self.account_id = account_id
        self.balance = balance

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented
        return self.account_id == other.account_id

    def __hash__(self):
        return hash(self.account_id)

**Making a collection its own iterator.** Returning `self` from `__iter__` and adding `__next__` works once and then breaks every subsequent loop, because the state has been consumed.

In [ ]:
# Wrong: collection is also its own iterator
class TransactionLog:
    def __init__(self):
        self.entries = []
        self._cursor = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._cursor >= len(self.entries):
            raise StopIteration
        entry = self.entries[self._cursor]
        self._cursor += 1
        return entry


log = TransactionLog()
log.entries.extend([("deposit", 100.0), ("withdraw", 30.0)])
list(log)                 # [('deposit', 100.0), ('withdraw', 30.0)]
list(log)                 # [] — exhausted

# Correct: return a fresh iterator each time
class TransactionLog:
    def __init__(self):
        self.entries = []

    def __iter__(self):
        return iter(self.entries)

**Raising `IndexError` from `__next__` instead of `StopIteration`.** The iteration machinery only catches `StopIteration`; any other exception propagates to the caller, terminating the loop with an error rather than ending it cleanly.

In [ ]:
# Wrong
def __next__(self):
    if self._cursor >= len(self.entries):
        raise IndexError       # propagates, breaks the loop
    ...

# Correct
def __next__(self):
    if self._cursor >= len(self.entries):
        raise StopIteration
    ...

**Overriding `__getattribute__` instead of `__getattr__`.** `__getattribute__` runs for every attribute access, including those used by the implementation itself. Forgetting to call `super().__getattribute__` produces infinite recursion.

In [ ]:
# Wrong: infinite recursion
class LazyAccount:
    def __getattribute__(self, name):
        if name not in self._loaded:       # this is itself an attribute access
            ...

# Correct: __getattr__ runs only when normal lookup fails
class LazyAccount:
    def __init__(self, loader):
        self._loader = loader
        self._loaded = None

    def __getattr__(self, name):
        if self._loaded is None:
            self._loaded = self._loader()
        return self._loaded[name]

**Using `__contains__` for an O(n) scan when a set would do.** `in` falls back to iteration if `__contains__` is not defined. For a list-backed container that is queried for membership often, defining `__contains__` against an internal `set` is a near-free speed-up.

In [ ]:
# Wrong: __contains__ inherited from list semantics; O(n) per call
class OwnerRegistry:
    def __init__(self, owners):
        self.owners = list(owners)

    def __contains__(self, owner):
        return owner in self.owners

# Correct: maintain a set internally for O(1) membership
class OwnerRegistry:
    def __init__(self, owners):
        self.owners = list(owners)
        self._index = set(owners)

    def __contains__(self, owner):
        return owner in self._index

**Suppressing exceptions accidentally from `__exit__`.** Returning a truthy value from `__exit__` swallows whatever exception propagated out of the `with` block. Returning `True` "just in case" hides bugs.

In [ ]:
# Wrong: swallows every exception that occurs inside the with block
def __exit__(self, exc_type, exc_value, traceback):
    self.cleanup()
    return True

# Correct: return None (or False) so exceptions continue to propagate
def __exit__(self, exc_type, exc_value, traceback):
    self.cleanup()
    return False

The right rule is to return a truthy value only when the manager is _intentionally_ acting as an exception handler, and to be specific about which exception type is being swallowed.